In [8]:
# Extrativo com "embeddings" via LSA (TF-IDF + SVD) — funciona offline
import re, textwrap
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.decomposition import TruncatedSVD
import numpy as np

# Como o texto completo já foi fornecido antes, vamos recolocá‑lo aqui
article_text = """
Enem 2025: Homem que revelou questões fala em 'coincidência' e diz que fez 'publicidade infeliz' para vender curso
PF apreendeu, na manhã deste sábado (23), o celular e o computador de Edcley Teixeira para investigá-lo.
Por Fantástico

23/11/2025 20h57  Atualizado há 9 horas

Enem 2025: Renata Ceribelli entrevista estudante alvo de investigação da PF
Enem 2025: Renata Ceribelli entrevista estudante alvo de investigação da PF

O Ministério da Educação (MEC) anulou três questões do segundo dia do Enem 2025 e acionou a Polícia Federal nesta semana, após o g1 revelar que o estudante Edcley Teixeira, de Sobral (Ceará), havia antecipado perguntas da prova em uma live.

Questionado pelo Fantástico, Edcley chamou de "coincidência" a similaridade entre as questões. Ele também admitiu ter pago alunos para memorizar perguntas de um pré-teste do MEC. "É uma forma de publicidade que hoje eu considero infeliz", afirmou ele, que cursa o quarto ano de Medicina.

Para investigar Edcley, a PF apreendeu na manhã deste sábado (23) aparelhos dele como celular e computador, além de documentos. As acusações de fraude começaram a circular na internet logo após o encerramento do segundo dia de provas do Enem, realizado em 16 de novembro, que reuniu quase cinco milhões de estudantes em todo o país.
A denúncia aponta que Edcley Teixeira, que ministra cursos online preparatórios para o Enem, adiantou em uma live para seus alunos (realizada na mesma semana do exame) perguntas "extremamente parecidas" com as que entraram na prova oficial.

Questões quase idênticas
A jornalista Luiza Tenente, do g1, mostrou ao Fantástico trechos da live de Edcley, vídeo que agora está com acesso fechado ao público. Na apuração, a jornalista encontrou cinco questões com bastante similaridade.

"Eu vi que eram os mesmos números, a mesma situação-problema, muitas vezes as alternativas eram basicamente idênticas", afirmou Luiza.
Edcley Teixeira negou que soubesse que as questões iriam entrar no exame: “Eu acho que essas similaridades pontuais foram coincidências”. A repórter Luiza Tenente, que publicou a primeira reportagem sobre o caso, disse: "É muito difícil acreditar que tenha sido sorte. Eu diria que é impossível."

Pré-testes
O estudante confessou que as perguntas anuladas pelo Enem estavam em um concurso que ele participou em 2024, o Prêmio Capes Talento Universitário, promovido pelo MEC. "Eu desconfiei que pudesse ser um tipo de pré-teste", ele afirmou, após dizer que teria identificado naquela prova padrões semelhantes aos do Enem.

Os chamados Pré-testes são provas que o Inep (Instituto Nacional de Estudos e Pesquisas Educacionais Anísio Teixeira) cria para testar perguntas que provavelmente cairão no Enem. Eles são aplicados em estudantes do terceiro ano do ensino médio com o objetivo de avaliar o conhecimento dos alunos.

O Inep confirmou que o prêmio serve como uma espécie de laboratório para validar perguntas que podem, ou não, cair no Enem.
A jornalista Luiza Tenente descobriu que Edcley pagava candidatos para memorizar perguntas dessa prova. Em mensagens, ele pedia detalhes de "até dez questões" do exame.

Durante a entrevista, Edcley disse que não vê "má-fé" nessa prática. "Não existia nenhum termo de compromisso, não existia nenhum termo de sigilo, não existia nem um edital".

Especialistas criticaram a vulnerabilidade do Banco Nacional de Itens (BNI), onde o Inep armazena as questões do Enem. Maria Helena Castro, ex-presidente do Inep, afirmou que a falta de um banco de itens mais robusto (com 100 mil ou 110 mil itens calibrados) faz com que o pré-teste seja necessário todo ano.

Já o atual presidente do Inep, Manuel Palacios, negou que haja alguma vulnerabilidade no sistema de pré-teste.

O Ministro da Educação, Camilo Santana, garantiu que o Enem deste ano não será anulado, reforçando que o exame é "um patrimônio do Brasil".
"""

def split_sentences(text):
    parts = re.split(r'(?<=[\.\!\?])\s+(?=[A-ZÀ-Ý"])', text)
    return [s.strip() for s in parts if len(s.split()) > 4]

sentences = split_sentences(article_text)

# TF‑IDF dos textos
vectorizer = TfidfVectorizer(min_df=1, stop_words=None)
X = vectorizer.fit_transform(sentences)

# LSA — reduz para embeddings semânticos
svd = TruncatedSVD(n_components=100, random_state=0)
X_emb = svd.fit_transform(X)

# embedding global do texto: média
doc_emb = X_emb.mean(axis=0, keepdims=True)

# similaridade cosine
def cosine(a,b): 
    return (a*b).sum() / (np.linalg.norm(a)*np.linalg.norm(b)+1e-9)

scores = [cosine(X_emb[i], doc_emb) for i in range(len(sentences))]

# selecionar top‑k
k = 4
top_idx = np.argsort(scores)[-k:]
top_idx = sorted(top_idx)

summary = " ".join(sentences[i] for i in top_idx)

print("=== Resumo extrativo com embeddings (LSA) ===\n")
print(textwrap.fill(summary, width=110))


=== Resumo extrativo com embeddings (LSA) ===

Enem 2025: Homem que revelou questões fala em 'coincidência' e diz que fez 'publicidade infeliz' para vender
curso PF apreendeu, na manhã deste sábado (23), o celular e o computador de Edcley Teixeira para investigá-lo.
As acusações de fraude começaram a circular na internet logo após o encerramento do segundo dia de provas do
Enem, realizado em 16 de novembro, que reuniu quase cinco milhões de estudantes em todo o país. A denúncia
aponta que Edcley Teixeira, que ministra cursos online preparatórios para o Enem, adiantou em uma live para
seus alunos (realizada na mesma semana do exame) perguntas "extremamente parecidas" com as que entraram na
prova oficial. O Inep confirmou que o prêmio serve como uma espécie de laboratório para validar perguntas que
podem, ou não, cair no Enem.
